# Eksplorasi data

Notebook ini memeriksa struktur data, kualitas kolom, distribusi target, dan ringkasan fitur sebelum pemodelan. Metrik kompetisi adalah Macro F1 untuk lima kelas `damage_grade`. Letakkan `train.csv`, `test.csv`, dan `sample_submission.csv` di folder `data/` sebelum menjalankan notebook.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / 'data'
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print(f'Train: {train.shape}; test: {test.shape}; sample: {sample.shape}')
display(train.head())

## Pemeriksaan skema dan ID

`building_id` dipertahankan untuk submission dan tidak dipakai sebagai label. Kolom fitur train dan test harus cocok setelah kolom target dilepas; urutan ID test harus sama dengan template submission.

In [ ]:
target = 'damage_grade'
assert target in train.columns
train_features = train.drop(columns=[target])
print('Kolom hanya di train:', sorted(set(train_features.columns) - set(test.columns)))
print('Kolom hanya di test:', sorted(set(test.columns) - set(train_features.columns)))
assert set(train_features.columns) == set(test.columns)
assert sample.columns.tolist() == ['building_id', 'damage_grade']
assert test['building_id'].equals(sample['building_id']), 'Urutan ID sample dan test tidak sama.'
print('ID test dan template cocok:', len(test), 'baris')

## Nilai hilang dan tipe fitur

In [ ]:
missing = train.isna().sum().sort_values(ascending=False)
display(missing[missing.gt(0)].head(20).rename('missing_count').to_frame())
print('Jumlah fitur numerik:', len(train.select_dtypes(include='number').columns))
print('Jumlah fitur kategorikal:', len(train.select_dtypes(include=['object', 'category']).columns))

display(train[target].value_counts().sort_index().rename('jumlah').to_frame())
display(train[target].value_counts(normalize=True).sort_index().rename('proporsi').to_frame())
numeric_features = train.drop(columns=[target, 'building_id'], errors='ignore').select_dtypes(include='number').columns
train[numeric_features].describe().T


In [ ]:
display(train[target].value_counts().sort_index().rename('jumlah').to_frame())
display(train[target].value_counts(normalize=True).sort_index().rename('proporsi').to_frame())
train.select_dtypes(include='number').describe().T

## Ringkasan kategori

Tabel berikut bersifat deskriptif. Semua transformasi dan statistik target untuk pemodelan harus dipelajari hanya dari bagian train di dalam split/fold.

In [ ]:
categorical_columns = train.select_dtypes(include=['object', 'category']).columns.tolist()
for column in categorical_columns[:8]:
    print(f'\n{column}')
    display(pd.crosstab(train[column], train[target], normalize='index').head(12))